# Pet claim files

**What this notebook is for:** every file uploaded to a pet claim, not only patient histories. The aim is
to say what kind of document each file is, summarise it, and group the files of one kind together.

**Kernel:** use `../vet-record-reader/.venv/bin/python`, the environment of the first notebook; it has
every package needed, so there is no second environment.

**Before the first run:** `gcloud auth application-default login` (signs this machine in to Google Cloud).

In [ ]:
import pandas as pd
from google.cloud import bigquery

# BigQuery client (your own Google login) and a query helper that dry-runs first
bq_client = bigquery.Client(project='hedvig-coruscant')


def run_query(sql, max_gb=10):
    dry = bq_client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    gb = dry.total_bytes_processed / 1e9
    print(f'Dry run: this query scans {gb:.2f} GB')
    if gb > max_gb:
        raise ValueError(f'{gb:.2f} GB is above max_gb={max_gb}. Approve by raising max_gb.')
    return bq_client.query(sql).result().to_dataframe()

In [ ]:
# Emma's query, with three changes agreed 18 Nov: claims_id/bucket/file_name added, all three deleted
# columns required empty, and a fixed random order so the same 30 files come back each run
SQL = """
SELECT f.id, f.claims_id, f.bucket, f.key, f.file_name, f.content_type, f.uploaded_at,
       f.is_member_accessible, f.display_name, f.upload_source
FROM hedvig-coruscant.dbt_prod.stg_claims_service__claim_file f
JOIN (
SELECT DISTINCT claim_id
FROM hedvig-coruscant.dbt_prod.claims_reporting
WHERE promise_category = 'Pet'
AND acquisition_source != 'manypets'
) cr ON cr.claim_id = f.claims_id
WHERE DATE(f.uploaded_at) > '2025-01-01'
AND f.deleted_at IS NULL
AND f.marked_as_deleted_at IS NULL
AND f.markd_as_deleted_at IS NULL
ORDER BY FARM_FINGERPRINT(CAST(f.id AS STRING))
LIMIT 30
"""
pet_files = run_query(SQL)
pet_files

In [ ]:
# Counts over all pet claim files since 2025: what values content_type, category, folder, upload_source
# and the three deleted columns take, with files and claims per value
SQL = """
WITH pet_files AS (
  SELECT f.*
  FROM `hedvig-coruscant.dbt_prod.stg_claims_service__claim_file` f
  JOIN (SELECT DISTINCT claim_id FROM `hedvig-coruscant.dbt_prod.claims_reporting`
        WHERE promise_category = 'Pet' AND acquisition_source != 'manypets') cr
    ON cr.claim_id = f.claims_id
  WHERE DATE(f.uploaded_at) > '2025-01-01'
)
SELECT 'content_type' AS what, content_type AS value, COUNT(*) AS files, COUNT(DISTINCT claims_id) AS claims FROM pet_files GROUP BY 2
UNION ALL SELECT 'category', category, COUNT(*), COUNT(DISTINCT claims_id) FROM pet_files GROUP BY 2
UNION ALL SELECT 'folder', folder, COUNT(*), COUNT(DISTINCT claims_id) FROM pet_files GROUP BY 2
UNION ALL SELECT 'upload_source', upload_source, COUNT(*), COUNT(DISTINCT claims_id) FROM pet_files GROUP BY 2
UNION ALL SELECT 'deleted flags',
  CONCAT('deleted_at=', CAST(deleted_at IS NOT NULL AS STRING),
         ' marked=', CAST(marked_as_deleted_at IS NOT NULL AS STRING),
         ' markd=', CAST(markd_as_deleted_at IS NOT NULL AS STRING)),
  COUNT(*), COUNT(DISTINCT claims_id) FROM pet_files GROUP BY 2
ORDER BY what, files DESC
"""
file_counts = run_query(SQL)
pd.set_option("display.max_rows", 200)
file_counts
